<center>

# Universidad Nacional de Lomas de Zamora
## Facultad de Ingeniería
### Proyecto FINAL - Estacion de calidad por Perfilómetria
#### Alumno: QUINTANA, Fernando Miguel

</center>

In [ ]:
# --------------------------------
# Importamos librerías
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Cargamos los archivos de las matrices de posición X, Y y Z
mtx_xi = np.loadtxt("mtx_x_izq.txt")  # Matriz pos. pixeles en X
mtx_yi = np.loadtxt("mtx_y_izq.txt")  # Matriz pos. pixeles en Y
mtx_zi = np.loadtxt("mtx_z_izq.txt")  # Matriz pos. pixeles en Z

# Creamos las variables auxiliares a utilizar en la ejecución del programa
scale_factor_x = 1     # Escalado en X
scale_factor_y = 2.3   # Escalado en Y
scale_factor_z = 2.3   # Escalado en Z


# Creamos los offset en X,  Y y Z para mover el grafico.
offset_x = 0
offset_y = -18
offset_z = 0

# Creamos la función para filtrado de "ROJO" mediante HSV
def HSV_FILTRO(image):
    # Duplicamos la imagen
    result = image.copy()
    # Convertimos la imagen a espacio de color HSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # Definimos los límites inferiores y superiores del color rojo en HSV
    lower_red = np.array([0, 50, 50])
    upper_red = np.array([10, 255, 255])
    # Creamos la máscara para el color rojo
    mask1 = cv2.inRange(hsv_image, lower_red, upper_red)
    # Definimos un rango adicional para el color rojo debido a su disposición en el espacio de color HSV
    lower_red = np.array([170, 50, 50])
    upper_red = np.array([180, 255, 255])
    # Creamos otra máscara para el color rojo
    mask2 = cv2.inRange(hsv_image, lower_red, upper_red)
    # Combinamos ambas máscaras para obtener una única máscara para el rojo
    mask = cv2.bitwise_or(mask1, mask2)
    # Aplicamos la máscara a la imagen original para obtener solo los píxeles rojos
    result = cv2.bitwise_and(result, result, mask=mask)
    # Convertimos la imagen resultante a escala de grises
    img_gris = cv2.cvtColor(result, cv2.COLOR_BGR2GRAY)
    # Retornamos la imagen filtrada
    return img_gris

# Creamos la función para calibración de imagen
def CALIBRAR(imagen, h, w):
    # PROPIEDADES INTRÍNSECAS DE CÁMARA
    # Matriz intrínseca de Cámara
    M_int = np.array([[1.38106997e03, 0.00000000e00, 6.87317037e02],
                      [0.00000000e00, 1.38209804e03, 3.82421364e02],
                      [0.00000000e00, 0.00000000e00, 1.00000000e00]])

    # Matríz de Distorsión
    D = np.array([8.30863128e-02, 1.57413825e00, 4.52467136e-04, 5.90615097e-03, -7.07611874e00])

    # Matriz Transformación de Cámara - Pixel
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(M_int, D, (w, h), 1, (w, h), centerPrincipalPoint=False)

    # Matriz Imagen calibrada
    dst = cv2.undistort(imagen, M_int, D, None, newcameramtx)

    # Se ajusta tamaño de imagen
    x, y, w, h = roi
    dst = dst[y : y + h, x : x + w]

    # Retorna imagen calibrada
    return dst

# Creamos la función para procesar la imagen
def PROCESAR(image):
    # Inicializamos las variables a utilizar
    perfil_puntos = []  # Lista de puntos del perfil de la pieza
    pos_pix2 = []  # Lista de cada pixel a encontrar en las matrices de referencia

    # Obtenemos el tamaño de la imagen recibida
    h, w = image.shape[:2]

    # Procesamos la imagen -> calibramos y filtramos "ROJO"
    image = CALIBRAR(image, h, w)
    image = HSV_FILTRO(image)

    # Aplicamos un tresh para poner los pixeles en 1 o 0 según los valores límites fijados
    image = cv2.threshold(image, 15, 255, cv2.THRESH_BINARY)[1]

    # Mostramos imagen con umbral aplicado
    cv2.imshow("Imagen", image)
    cv2.waitKey(0)

    # Recorremos todas las columnas de la imagen, para asegurarnos obtener una linea lo más fina posible a la curva del laser
    img_columnas = len(image[0])  # Obtenemos la cantidad de columnas que tiene la imagen

    for col in range(0, img_columnas):
        vpx = np.where(image[:, col] != 0)  # Aquellos pixeles con "1" son los pixeles que queremos ubicar

        # Si para dicha columna tenemos pixeles en "1", entonces lo guardamos en la lista
        if len(vpx[0]) > 0:
            # Ubicamos el pixel más centrado de la linea para ajustar lo más posible.
            if len(vpx[0]) % 2 == 0:
                pos_pix2.append([vpx[0][int(len(vpx[0]) / 2)], col])
            else:
                pos_pix2.append([vpx[0][int(round(len(vpx[0]) / 2, 0))], col])

    # Para cada posición encontrada, se obtienen los valores de X, Y y Z en el espacio según sistema de referencia
    for px in pos_pix2:
        # Posición en X * escalado
        x = (mtx_xi[px[0], px[1]] * scale_factor_x) + offset_x
        # Posición en Y * escalado
        y = (mtx_yi[px[0], px[1]] * scale_factor_y) + offset_y
        # Posición en Z * escalado
        z = (mtx_zi[px[0], px[1]] * scale_factor_z) + offset_z

        # Guardamos en la lista la posición en X, Y y Z de cada pixel
        perfil_puntos.append([x, y, z])

    # Retornamos la lista de puntos
    return np.array(perfil_puntos)

# Leemos imagen del directorio
img = cv2.imread("D:/Trabajo Final (Perfilometro)/Programa/ImgsPerfilometro/WIN_20240208_21_51_56_Pro.jpg")
#img = cv2.imread("D:/Trabajo Final (Perfilometro)/Programa/ImgsPerfilometro/WIN_20240218_16_16_26_Pro.jpg")

# Mostramos imagen original
cv2.imshow("Original", img)
cv2.waitKey(0)

# Procesamos imagen y obtenemos el perfil
puntos_perfil = PROCESAR(img)

# Graficamos los puntos obtenidos
plt.figure(figsize=(20, 10))  # Ajustamos el tamaño del gráfico
plt.plot(puntos_perfil[:, 2], puntos_perfil[:, 1], linestyle="dashed", color="red")
plt.xticks(range(-15, 35, 1))  # Ajustamos los intervalos de los números en los ejes x y y
plt.yticks(range(-5, 30, 1))
plt.grid()
plt.xlabel("Ancho [mm]")
plt.ylabel("Alto [mm]")
plt.xlim(-15, 35)
plt.ylim(-5, 30)
plt.show()





### Calculo de altura del perfil

In [ ]:
# Calculamos la altura
puntos_encima_2mm = puntos_perfil[puntos_perfil[:, 1] > 2]  # Filtramos los puntos por encima de 2 mm en el eje Y
puntos_debajo_2mm = puntos_perfil[puntos_perfil[:, 1] < 2]  # Filtramos los puntos por debajo de 2 mm en el eje Y

# Ordenamos los puntos filtrados por su valor en el eje X
puntos_encima_2mm_sorted = np.sort(puntos_encima_2mm, axis=0)
puntos_debajo_2mm_sorted = np.sort(puntos_debajo_2mm, axis=0)

# Calculamos los índices que representan el 10% de los valores más bajos y altos para los puntos por encima de 2 mm
n_encima = len(puntos_encima_2mm_sorted)
low_index_encima = int(n_encima * 0.1)
high_index_encima = int(n_encima * 0.9)

# Seleccionamos solo los valores dentro del rango del 10% al 90% para los puntos por encima de 2 mm
puntos_80_percent_encima = puntos_encima_2mm_sorted[low_index_encima:high_index_encima]

# Calculamos el promedio de los valores seleccionados para los puntos por encima de 2 mm
altura_promedio_80_percent_encima = np.mean(puntos_80_percent_encima[:, 1])

# Realizamos el mismo proceso para los puntos por debajo de 2 mm
n_debajo = len(puntos_debajo_2mm_sorted)
low_index_debajo = int(n_debajo * 0.1)
high_index_debajo = int(n_debajo * 0.9)
puntos_80_percent_debajo = puntos_debajo_2mm_sorted[low_index_debajo:high_index_debajo]
altura_promedio_80_percent_debajo = np.mean(puntos_80_percent_debajo[:, 1])

# Calculamos la diferencia entre los promedios de los puntos por encima y por debajo de 2 mm
diferencia_altura_promedio = round(altura_promedio_80_percent_encima - altura_promedio_80_percent_debajo, 1)

# Mostramos los resultados
print("Altura promedio del perfil (80% de los valores por encima de 2 mm):", round(altura_promedio_80_percent_encima, 1), "mm")
print("Altura promedio de la base (80% de los valores por debajo de 2 mm):", round(altura_promedio_80_percent_debajo, 1), "mm")
print("Diferencia de alturas promedio:", diferencia_altura_promedio, "mm")


### Calculo del ancho del perfil

In [ ]:
# Encontrar el inicio y el final del perfil
inicio_perfil = None
fin_perfil = None
for i in range(len(puntos_perfil)):
    # Si el valor de Y es mayor a 2 mm, encontramos el inicio del perfil
    if puntos_perfil[i, 1] > 2:
        inicio_perfil = i
        break

for i in range(len(puntos_perfil) - 1, -1, -1):
    # Si el valor de Y es mayor a 2 mm, encontramos el final del perfil
    if puntos_perfil[i, 1] > 2:
        fin_perfil = i
        break

# Calcular el ancho del perfil
if inicio_perfil is not None and fin_perfil is not None:
    ancho_perfil = puntos_perfil[inicio_perfil, 2] - puntos_perfil[fin_perfil, 2] 
    print("Ancho del perfil:", ancho_perfil, "mm")
else:
    print("No se encontraron puntos para calcular el ancho del perfil.")

### Graficamos el perfil con los datos obtenidos

In [ ]:
# Graficar el perfil con datos:
plt.figure(figsize=(20, 10))  # Ajustamos el tamaño del gráfico
plt.plot(puntos_perfil[:, 2], puntos_perfil[:, 1], linestyle="dashed", color="red")

# Dibujar flecha para el ancho
plt.annotate('', xy=(puntos_perfil[fin_perfil, 2], altura_promedio_80_percent_encima + 2),
             xytext=(puntos_perfil[inicio_perfil, 2], altura_promedio_80_percent_encima + 2),
             arrowprops=dict(arrowstyle='<->', color='green'))
plt.text((puntos_perfil[inicio_perfil, 2] + puntos_perfil[fin_perfil, 2]) / 2,
         altura_promedio_80_percent_encima + 2.5, 'Ancho del perfil Promedio: {} mm'.format(round(ancho_perfil, 1)),
         color='green', fontsize=12, ha='center')

# Dibujar flecha para la altura promedio
plt.annotate('', xy=(0, altura_promedio_80_percent_debajo),
             xytext=(0, altura_promedio_80_percent_encima),
             arrowprops=dict(arrowstyle='<->', color='blue'))
plt.text(0.5, (altura_promedio_80_percent_encima + altura_promedio_80_percent_debajo) / 2,
         f'Diferencia de alturas promedio: {diferencia_altura_promedio} mm', color='blue', fontsize=12, ha='left')

plt.xticks(range(-15, 35, 1))  # Ajustamos los intervalos de los números en los ejes x y y
plt.yticks(range(-5, 30, 1))
plt.grid()
plt.xlabel("Ancho [mm]")
plt.ylabel("Alto [mm]")
plt.xlim(-15, 35)
plt.ylim(-5, 30)
plt.show()

### INTERFACE ARDUINO

In [ ]:
import serial
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Configurar puerto serie (Cambia 'COM4' si es necesario)
ser = serial.Serial('COM4', 115200, timeout=1)  # Windows: 'COM4', Linux/Mac: '/dev/ttyUSB0'
time.sleep(2)  # Esperar a que el puerto se estabilice

print("Esperando comandos de Arduino...")

#--------------------------------------------------------------------------------------------

# Importamos librerías

# Cargamos los archivos de las matrices de posición X, Y y Z
mtx_xi = np.loadtxt("mtx_x_izq.txt")  # Matriz pos. pixeles en X
mtx_yi = np.loadtxt("mtx_y_izq.txt")  # Matriz pos. pixeles en Y
mtx_zi = np.loadtxt("mtx_z_izq.txt")  # Matriz pos. pixeles en Z

#--------------------------------------------------------------------------------------------

# Creamos las variables auxiliares a utilizar en la ejecución del programa
scale_factor_x = 1     # Escalado en X
scale_factor_y = 2.3   # Escalado en Y
scale_factor_z = 2.3   # Escalado en Z

#--------------------------------------------------------------------------------------------

# Creamos los offset en X,  Y y Z para mover el grafico.
offset_x = 0
offset_y = -18
offset_z = 0

#--------------------------------------------------------------------------------------------

# Creamos la función para filtrado de "ROJO" mediante HSV
def HSV_FILTRO(image):
    # Duplicamos la imagen
    result = image.copy()
    # Convertimos la imagen a espacio de color HSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # Definimos los límites inferiores y superiores del color rojo en HSV
    lower_red = np.array([0, 50, 50])
    upper_red = np.array([10, 255, 255])
    # Creamos la máscara para el color rojo
    mask1 = cv2.inRange(hsv_image, lower_red, upper_red)
    # Definimos un rango adicional para el color rojo debido a su disposición en el espacio de color HSV
    lower_red = np.array([170, 50, 50])
    upper_red = np.array([180, 255, 255])
    # Creamos otra máscara para el color rojo
    mask2 = cv2.inRange(hsv_image, lower_red, upper_red)
    # Combinamos ambas máscaras para obtener una única máscara para el rojo
    mask = cv2.bitwise_or(mask1, mask2)
    # Aplicamos la máscara a la imagen original para obtener solo los píxeles rojos
    result = cv2.bitwise_and(result, result, mask=mask)
    # Convertimos la imagen resultante a escala de grises
    img_gris = cv2.cvtColor(result, cv2.COLOR_BGR2GRAY)
    # Retornamos la imagen filtrada
    return img_gris

#--------------------------------------------------------------------------------------------

# Creamos la función para calibración de imagen
def CALIBRAR(imagen, h, w):
    # PROPIEDADES INTRÍNSECAS DE CÁMARA
    # Matriz intrínseca de Cámara
    M_int = np.array([[1.38106997e03, 0.00000000e00, 6.87317037e02],
                      [0.00000000e00, 1.38209804e03, 3.82421364e02],
                      [0.00000000e00, 0.00000000e00, 1.00000000e00]])

    # Matríz de Distorsión
    D = np.array([8.30863128e-02, 1.57413825e00, 4.52467136e-04, 5.90615097e-03, -7.07611874e00])

    # Matriz Transformación de Cámara - Pixel
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(M_int, D, (w, h), 1, (w, h), centerPrincipalPoint=False)

    # Matriz Imagen calibrada
    dst = cv2.undistort(imagen, M_int, D, None, newcameramtx)

    # Se ajusta tamaño de imagen
    x, y, w, h = roi
    dst = dst[y : y + h, x : x + w]

    # Retorna imagen calibrada
    return dst

#--------------------------------------------------------------------------------------------

# Creamos la función para procesar la imagen
def PROCESAR(image):
    # Inicializamos las variables a utilizar
    perfil_puntos = []  # Lista de puntos del perfil de la pieza
    pos_pix2 = []  # Lista de cada pixel a encontrar en las matrices de referencia

    # Obtenemos el tamaño de la imagen recibida
    h, w = image.shape[:2]

    # Procesamos la imagen -> calibramos y filtramos "ROJO"
    image = CALIBRAR(image, h, w)
    image = HSV_FILTRO(image)

    # Aplicamos un tresh para poner los pixeles en 1 o 0 según los valores límites fijados
    image = cv2.threshold(image, 15, 255, cv2.THRESH_BINARY)[1]

    # Recorremos todas las columnas de la imagen, para asegurarnos obtener una linea lo más fina posible a la curva del laser
    img_columnas = len(image[0])  # Obtenemos la cantidad de columnas que tiene la imagen

    for col in range(0, img_columnas):
        vpx = np.where(image[:, col] != 0)  # Aquellos pixeles con "1" son los pixeles que queremos ubicar

        # Si para dicha columna tenemos pixeles en "1", entonces lo guardamos en la lista
        if len(vpx[0]) > 0:
            # Ubicamos el pixel más centrado de la linea para ajustar lo más posible.
            if len(vpx[0]) % 2 == 0:
                pos_pix2.append([vpx[0][int(len(vpx[0]) / 2)], col])
            else:
                pos_pix2.append([vpx[0][int(round(len(vpx[0]) / 2, 0))], col])

    # Para cada posición encontrada, se obtienen los valores de X, Y y Z en el espacio según sistema de referencia
    for px in pos_pix2:
        # Posición en X * escalado
        x = (mtx_xi[px[0], px[1]] * scale_factor_x) + offset_x
        # Posición en Y * escalado
        y = (mtx_yi[px[0], px[1]] * scale_factor_y) + offset_y
        # Posición en Z * escalado
        z = (mtx_zi[px[0], px[1]] * scale_factor_z) + offset_z

        # Guardamos en la lista la posición en X, Y y Z de cada pixel
        perfil_puntos.append([x, y, z])

    # Retornamos la lista de puntos
    return np.array(perfil_puntos)

#--------------------------------------------------------------------------------------------

# Esperar hasta que Arduino envíe "MEDIR"
while True:
    if ser.in_waiting > 0:
        comando = ser.readline().decode('utf-8').strip()
        print(f"Recibido comando: {comando}")

        if comando == "MEDIR":
            try:
                
                # Capturar imagen desde la cámara
                cap = cv2.VideoCapture(0)  # Usar cámara por defecto (ajustar índice si es necesario)
                cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
                cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

                if not cap.isOpened():
                    print("Error: No se pudo abrir la cámara.")
                    ser.write("0 0\r\n".encode('utf-8'))
                    continue

                ret, frame = cap.read()
                cap.release()

                if not ret:
                    print("Error: No se pudo capturar la imagen.")
                    ser.write("0 0\r\n".encode('utf-8'))
                    continue

                # Guardar imagen capturada como 'profile.jpg', sobrescribiendo si ya existe
                ruta_imagen = "D:/UNLZ/4 - Proyecto final (1C2025)/Phyton/Programa/ImgsPerfilometro/profile.jpg"
                cv2.imwrite(ruta_imagen, frame)

                # Cargar la imagen recién guardada para procesarla
                img = cv2.imread(ruta_imagen)

                
                # Procesa imagen SOLO cuando Arduino lo solicite
                #img = cv2.imread("D:/Trabajo Final (Perfilometro)/Programa/ImgsPerfilometro/WIN_20240208_21_51_56_Pro.jpg")
                #img = cv2.imread("D:/Trabajo Final (Perfilometro)/Programa/ImgsPerfilometro/WIN_20240218_16_16_26_Pro.jpg")
                
                if img is None:
                    print("Error: No se pudo cargar la imagen.")
                    ser.write("0 0\r\n".encode('utf-8'))
                    continue
                
                # Procesamos imagen y obtenemos el perfil
                puntos_perfil = PROCESAR(img)
                
#--------------------------------------------------------------------------------------------
                
                # CALCULO ALTURA PERFIL
                puntos_encima_2mm = puntos_perfil[puntos_perfil[:, 1] > 2]  # Filtramos los puntos por encima de 2 mm en el eje Y
                puntos_debajo_2mm = puntos_perfil[puntos_perfil[:, 1] < 2]  # Filtramos los puntos por debajo de 2 mm en el eje Y

                # Ordenamos los puntos filtrados por su valor en el eje X
                puntos_encima_2mm_sorted = np.sort(puntos_encima_2mm, axis=0)
                puntos_debajo_2mm_sorted = np.sort(puntos_debajo_2mm, axis=0)

                # Calculamos los índices que representan el 10% de los valores más bajos y altos para los puntos por encima de 2 mm
                n_encima = len(puntos_encima_2mm_sorted)
                low_index_encima = int(n_encima * 0.1)
                high_index_encima = int(n_encima * 0.9)

                # Seleccionamos solo los valores dentro del rango del 10% al 90% para los puntos por encima de 2 mm
                puntos_80_percent_encima = puntos_encima_2mm_sorted[low_index_encima:high_index_encima]

                # Calculamos el promedio de los valores seleccionados para los puntos por encima de 2 mm
                altura_promedio_80_percent_encima = np.mean(puntos_80_percent_encima[:, 1])

                # Realizamos el mismo proceso para los puntos por debajo de 2 mm
                n_debajo = len(puntos_debajo_2mm_sorted)
                low_index_debajo = int(n_debajo * 0.1)
                high_index_debajo = int(n_debajo * 0.9)
                puntos_80_percent_debajo = puntos_debajo_2mm_sorted[low_index_debajo:high_index_debajo]
                altura_promedio_80_percent_debajo = np.mean(puntos_80_percent_debajo[:, 1])

                # Calculamos la diferencia entre los promedios de los puntos por encima y por debajo de 2 mm
                diferencia_altura_promedio = round(altura_promedio_80_percent_encima - altura_promedio_80_percent_debajo, 1)

#--------------------------------------------------------------------------------------------
                
                # CALCULO ANCHO PERFIL
                # Encontrar el inicio y el final del perfil
                inicio_perfil = None
                fin_perfil = None
                for i in range(len(puntos_perfil)):
                    # Si el valor de Y es mayor a 2 mm, encontramos el inicio del perfil
                    if puntos_perfil[i, 1] > 2:
                        inicio_perfil = i
                        break

                for i in range(len(puntos_perfil) - 1, -1, -1):
                    # Si el valor de Y es mayor a 2 mm, encontramos el final del perfil
                    if puntos_perfil[i, 1] > 2:
                        fin_perfil = i
                        break

                # Calcular el ancho del perfil
                if inicio_perfil is not None and fin_perfil is not None:
                    ancho_perfil = puntos_perfil[inicio_perfil, 2] - puntos_perfil[fin_perfil, 2] 
                           
#--------------------------------------------------------------------------------------------
                   
                # GRAFICO EL PERFIL
                
                plt.figure(figsize=(20, 10))  # Ajustamos el tamaño del gráfico
                plt.plot(puntos_perfil[:, 2], puntos_perfil[:, 1], linestyle="dashed", color="red")

                # Dibujar flecha para el ancho
                plt.annotate('', xy=(puntos_perfil[fin_perfil, 2], altura_promedio_80_percent_encima + 2),
                             xytext=(puntos_perfil[inicio_perfil, 2], altura_promedio_80_percent_encima + 2),
                             arrowprops=dict(arrowstyle='<->', color='green'))
                plt.text((puntos_perfil[inicio_perfil, 2] + puntos_perfil[fin_perfil, 2]) / 2,
                         altura_promedio_80_percent_encima + 2.5, 'Ancho del perfil Promedio: {} mm'.format(round(ancho_perfil, 1)),
                         color='green', fontsize=12, ha='center')

                # Dibujar flecha para la altura promedio
                plt.annotate('', xy=(0, altura_promedio_80_percent_debajo),
                             xytext=(0, altura_promedio_80_percent_encima),
                             arrowprops=dict(arrowstyle='<->', color='blue'))
                plt.text(0.5, (altura_promedio_80_percent_encima + altura_promedio_80_percent_debajo) / 2,
                         f'Diferencia de alturas promedio: {diferencia_altura_promedio} mm', color='blue', fontsize=12, ha='left')

                plt.xticks(range(-15, 35, 1))  # Ajustamos los intervalos de los números en los ejes x y y
                plt.yticks(range(-5, 30, 1))
                plt.grid()
                plt.xlabel("Ancho [mm]")
                plt.ylabel("Alto [mm]")
                plt.xlim(-15, 35)
                plt.ylim(-5, 30)
                plt.show()
                    
#--------------------------------------------------------------------------------------------

                # ENVIO VALORES A ARDUNIO
                mensaje = f"{ancho_perfil} {diferencia_altura_promedio}\r\n"
                ser.write(mensaje.encode('utf-8'))
                print(f"Enviando medición: Ancho={ancho_perfil}, Alto={diferencia_altura_promedio}")

            except Exception as e:
                print(f"Error al calcular perfil: {e}")
                ser.write("0 0\r\n".encode('utf-8'))
                
                